# Proyecto Unificado: Matrimonios y Divorcios 2020
Este notebook combina limpieza, generación de variables, análisis descriptivo, visualizaciones y clustering con KMeans (sin usar PCA).

In [ ]:
# Importar librerías principales
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Cargar datos
df = pd.read_csv(r'/workspaces/Analisis-Matrimonios-y-Divorcios-2020---Proyecto-FCDIA-/ProeyctoFCDIA/EDV_2020.csv', sep=';')

In [ ]:
# --- 1. Limpieza y preparación ---

def limpiar_sentinela(df, columnas, valor):
    df[columnas] = df[columnas].replace(valor, pd.NA)

sentinelas = {
    99: [
        'niv_inst2','hijos_2','dia_nac2','mes_nac2',
        'niv_inst1','hijos_1','dia_nac1','cau_div','mes_nac1'
    ],
    9:  ['mcap_bie','nac_1','nac_2','sabe_leer1','sabe_leer2'],
    999: ['edad_1','edad_2'],
    9999: ['anio_nac1','anio_nac2']
}

for valor, columnas in sentinelas.items():
    limpiar_sentinela(df, columnas, valor)

columnas_fechas = ['fecha_insc','fecha_nac2','fecha_nac1','fecha_div','fecha_mat']
df[columnas_fechas] = df[columnas_fechas].replace(' ', pd.NA)

df = df.dropna().reset_index(drop=True)

In [ ]:
# --- 2. Generación de variables ---
# Convertir a numérico y crear fecha de matrimonio
cols_fecha = ['dia_mat', 'mes_mat', 'anio_mat']
df[cols_fecha] = df[cols_fecha].apply(pd.to_numeric, errors='coerce')

df['fecha_mat'] = pd.to_datetime(
    df[['anio_mat', 'mes_mat', 'dia_mat']]
      .rename(columns={'anio_mat': 'year','mes_mat': 'month','dia_mat': 'day'}),
    errors='coerce'
)

df['dia_sem_mat'] = df['fecha_mat'].dt.weekday + 1

# Variables de pareja
df['total_hijos'] = df['hijos_1'] + df['hijos_2']
df['nivel_pareja'] = df[['niv_inst1','niv_inst2']].mean(axis=1)
# Diferencia de nivel (mayor - menor)
df['dif_nivel_inst'] = df[['niv_inst1','niv_inst2']].max(axis=1) - df[['niv_inst1','niv_inst2']].min(axis=1)

## Inspección rápida

In [ ]:
print(df.columns)
df.head()

## Análisis por día, mes y fin de semana

In [ ]:
# Día de la semana
sns.countplot(data=df, x='dia_sem_mat')
plt.title('Registros por día de la semana')
plt.show()

# Boxplot por día
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='dia_sem_mat', y='dur_mat')
plt.title('Duración del matrimonio por día de la semana')
plt.show()

# Por fin de semana
df['tipo_dia'] = df['dia_sem_mat'].apply(lambda x: 'Lunes-Viernes' if x in [1,2,3,4,5] else 'Sabado-Domingo')
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='tipo_dia', y='dur_mat')
plt.title('Duración por tipo de día')
plt.show()

## Educación e hijos

In [ ]:
# Parejas con mismo nivel educativo
df_mismo = df[df['niv_inst1'] == df['niv_inst2']].copy()
df_mismo['categoria_edu'] = df_mismo['niv_inst1'].apply(lambda n: 'Hasta educación básica' if 1<=n<=4 else ('Hasta Posgrado' if 5<=n<=9 else 'Otro'))
df_filtrado = df_mismo[df_mismo['categoria_edu'].isin(['Hasta educación básica','Hasta Posgrado'])]

# Estadísticas
for cat in ['Hasta educación básica','Hasta Posgrado']:
    s = df_filtrado[df_filtrado['categoria_edu']==cat]
    print(cat, 'n=', len(s), 'media hijos=', s['total_hijos'].mean())

# Visualización simple
plt.figure(figsize=(8,5))
sns.boxplot(data=df_filtrado, x='categoria_edu', y='total_hijos')
plt.title('Número de hijos por categoría educativa (parejas mismo nivel)')
plt.show()

## Educación cruzada (heatmap)

In [ ]:
contingency = pd.crosstab(df['niv_inst1'], df['niv_inst2'])
plt.figure(figsize=(10,8))
sns.heatmap(contingency, annot=False, cmap='viridis')
plt.title('Matriz niv_inst1 vs niv_inst2')
plt.show()

## Filtrado por cantidad de hijos

In [ ]:
# Ejemplo: parejas con 1 hijo
h = 1
subset = df[df['total_hijos']==h]
print('Parejas con', h, 'hijo(s):', len(subset))
if len(subset)>0:
    plt.figure(figsize=(8,5))
    sns.boxplot(data=subset, x=df[['niv_inst1','niv_inst2']].max(axis=1), y=subset['dur_mat'])
    plt.title(f'Duración por nivel (parejas con {h} hijo(s))')
    plt.show()

## Clustering con KMeans (sin PCA)

In [ ]:
# Selección de variables para clustering
features = ['edad_1','edad_2','dur_mat','total_hijos','nivel_pareja','dif_nivel_inst']
X = df[features].dropna().copy()
df_cluster = df.loc[X.index].copy()

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Elegir k (puedes modificar)
k = 4
kmeans = KMeans(n_clusters=k, random_state=3, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)

# Centroides (en escala original)
centroides = scaler.inverse_transform(kmeans.cluster_centers_)
df_centroides = pd.DataFrame(centroides, columns=features)

print('Clusters asignados:', df_cluster['cluster'].value_counts().to_dict())

In [ ]:
# Visualización: parejas de variables (sin PCA) y centroides
pairs_plot = [
    ('dur_mat', 'total_hijos'),
    ('dur_mat', 'nivel_pareja'),
    ('total_hijos', 'nivel_pareja')
]

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(18,5))
for ax, (x_var, y_var) in zip(axes, pairs_plot):
    sns.scatterplot(data=df_cluster, x=x_var, y=y_var, hue='cluster', palette='Set2', s=35, alpha=0.8, ax=ax, legend=False)
    ax.scatter(df_centroides[x_var], df_centroides[y_var], c='black', s=220, marker='X')
    ax.set_xlabel(x_var)
    ax.set_ylabel(y_var)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Cluster', loc='upper right', bbox_to_anchor=(0.98,0.9), frameon=False)
plt.suptitle(f'Clustering KMeans (k={k}) – Proyecciones 2D sin PCA')
plt.tight_layout(rect=[0,0,0.93,0.95])
plt.show()

## Resumen
- Clustering realizado con KMeans sobre variables seleccionadas.
- Visualizaciones muestran proyecciones en pares de variables (no se usó PCA).